# torch_embed

> The same embedding as the MAX engine, from the PyTorch wav2vec 2.0 weights.


In [ ]:
#| default_exp torch_embed


In [ ]:
#| hide
from nbdev.showdoc import *


The app embeds with Modular MAX and does not import this module. Training a head never needs it either: the examples in the database are already embeddings.

This is the PyTorch copy of that embedding, for a notebook that has the waveform. Install it with the `train` pixi environment (`pixi install -e train`), which is separate from the MAX environment. `embed_waveform` mean-pools the real frames, subtracts `embed_center.npy`, and L2-normalizes, including the pad up to a whole second that `Engine.embed` uses.


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np

from nadakit.config import base_960h_config
from nadakit.core import embed_center_path
from nadakit.features import normalize


In [ ]:
#| export
REPO = "facebook/wav2vec2-base-960h"
SAMPLE_RATE = 16_000


In [ ]:
#| export
def load_center() -> np.ndarray:
    """`embed_center.npy` as a 768-vector, or zeros when the file is absent."""
    path = embed_center_path()
    if path.is_file():
        return np.load(path).astype(np.float32).reshape(-1)
    return np.zeros(base_960h_config().hidden_size, np.float32)


In [ ]:
#| export
def load_wav2vec(repo: str = REPO, trainable: bool = False):
    """Hugging Face `Wav2Vec2Model` in eval mode. Weights require grad only when `trainable`."""
    import torch
    from transformers import Wav2Vec2Model

    model = Wav2Vec2Model.from_pretrained(repo)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(trainable)
    return model


In [ ]:
#| export
def embed_waveform(model, wave, center=None, *, max_seconds: int = 10) -> np.ndarray:
    """One clip, one embedding. Same reduction as `Engine.embed`."""
    import torch

    cfg = base_960h_config()
    x = np.asarray(wave, np.float32).reshape(-1)[: SAMPLE_RATE * max_seconds]
    if x.size < SAMPLE_RATE // 10:
        raise ValueError("That sound was too short. Hold it a little longer.")
    real = cfg.output_length(int(x.size))
    x = np.pad(normalize(x), (0, (-int(x.size)) % SAMPLE_RATE))
    with torch.no_grad():
        hidden = model(torch.from_numpy(np.ascontiguousarray(x))[None]).last_hidden_state
    center = load_center() if center is None else np.asarray(center, np.float32).reshape(-1)
    v = hidden[0, :real].mean(0).detach().cpu().numpy().astype(np.float32) - center
    return v / max(float(np.linalg.norm(v)), 1e-12)


In [ ]:
#| export
class TorchEmbed:
    """Frozen wav2vec 2.0 plus the centering the app uses."""

    def __init__(self, repo: str = REPO):
        self.model = load_wav2vec(repo)
        self.center = load_center()

    def embed(self, wave) -> np.ndarray:
        return embed_waveform(self.model, wave, self.center)


```python
from nadakit.torch_embed import TorchEmbed
embedder = TorchEmbed()
vector = embedder.embed(wave)  # 16 kHz float32
```

Leave the encoder frozen for the head in the next notebook. Fine-tuning it would move every stored embedding, and the library does not keep the waveforms those embeddings came from.
